# T4 — DVC: versionamento de dados

**Objetivo:** versionar dados como versionamos código, sem colocar os dados no Git, e ligar cada modelo
à versão exata dos dados que o treinou.

| Peça | Onde fica | Vai para o Git? |
|---|---|---|
| `bike.csv` (o dado) | sua pasta + cache `.dvc/cache` + **remote** | ❌ |
| `bike.csv.dvc` (ponteiro: hash md5 + tamanho) | ao lado do dado | ✅ |
| `.dvc/config` (onde fica o remote) | `.dvc/` | ✅ |

> 💡 O DVC e o remote (`../dvc-storage`) foram configurados na T1. Lá, o pipeline só **anotava o hash** do
> `bike.csv` no `dvc.lock`. Agora o próprio dado passa a ser versionado.
>
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

In [2]:
%%bash
git status --short
git log --oneline | head -5
dvc remote list

 M aula01/t0_ambiente.ipynb
 M aula01/t4_dvc_dados.ipynb
 M data/raw/bike.csv.dvc
?? .vscode/
51b5732 Retreina pipeline com dados v2
cca8991 Dados v1: 2011
a5bb437 Código feito
3e93abe Dados v1: 2011
55b55a3 HGB: max_leaf_nodes 15
local   /workspaces/dvc-storage (default)


## 1. Versionar os dados v1 (2011)

São sempre os mesmos 4 passos. **Guarde esta receita: você vai repeti-la na Parte 2.**

| Passo | Comando | O que faz |
|---|---|---|
| 1 | `dvc add data/raw/bike.csv` | calcula o md5, guarda no cache e cria o ponteiro `.dvc` |
| 2 | `git add data/raw/bike.csv.dvc` + `git commit` | o **ponteiro** vai para o Git |
| 3 | `git tag dados-v1` | um nome fácil para esta versão |
| 4 | `dvc push` | o **dado** vai para o remote |

In [3]:
%%bash
dvc add data/raw/bike.csv
cat data/raw/bike.csv.dvc

⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
outs:
- md5: fc7d423afbce5699a61ac1a8acc533b8
  size: 562742
  hash: md5
  path: bike.csv


O `.gitignore` do template já ignora CSVs dentro de `data/`, por isso o DVC não precisou criar um novo.

In [4]:
%%bash
git add data/raw/bike.csv.dvc
git commit -m "Dados v1: 2011"
git tag dados-v1
dvc push

[main 15dec4a] Dados v1: 2011
 1 file changed, 2 insertions(+), 2 deletions(-)


fatal: tag 'dados-v1' already exists


Everything is up to date.


## 2. Desastre: apagaram o dado **e** o cache

Se o dado está no remote, basta um `dvc pull`.

In [5]:
%%bash
rm -rf data/raw/bike.csv .dvc/cache
dvc pull
wc -l data/raw/bike.csv

A       data/raw/bike.csv
9 files fetched and 1 file added
8646 data/raw/bike.csv


## 3. Como voltar no tempo (você vai usar na Parte 2)

Para trazer uma versão antiga são **2 comandos**: o Git traz o **ponteiro** antigo e o
`dvc checkout` traz o **dado** correspondente.

```bash
git checkout <tag> -- data/raw/bike.csv.dvc   # ponteiro da versão <tag>
dvc checkout                                  # dado que corresponde ao ponteiro
```
Para voltar à versão atual, troque `<tag>` por `HEAD`.

---
# Parte 2 · Sozinhos

Chegaram dados novos: o **1º semestre de 2012**. Versione-os com a mesma receita da Parte 1.
Repare no `dvc status`: ele mostra que o dado mudou **e** que o estágio `prepare` do pipeline, que depende dele, ficou desatualizado.

In [6]:
%%bash
python src/get_data.py --until 2012-06-30
dvc status

data/raw/bike.csv: 13,003 linhas | 2011-01-01 a 2012-06-30
prepare:
	changed deps:
		modified:           data/raw/bike.csv
data/raw/bike.csv.dvc:
	changed outs:
		modified:           data/raw/bike.csv


### 1. Versione os dados v2 (receita do passo 1)

In [7]:
%%bash
git add data/raw/bike.csv
git commit -m "Dados v2: até jun/2012"
git tag -a dados-v2 -m "Dados v2: até jun/2012"
git log --oneline --decorate -5

The following paths are ignored by one of your .gitignore files:
data/raw/bike.csv
hint: Use -f if you really want to add them.
hint: Disable this message with "git config set advice.addIgnoredFile false"


On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   aula01/t0_ambiente.ipynb
	modified:   aula01/t4_dvc_dados.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.vscode/

no changes added to commit (use "git add" and/or "git commit -a")


fatal: tag 'dados-v2' already exists


15dec4a (HEAD -> main) Dados v1: 2011
51b5732 Retreina pipeline com dados v2
cca8991 (tag: dados-v2) Dados v1: 2011
a5bb437 Código feito
3e93abe (tag: dados-v1) Dados v1: 2011


### 2. Viagem no tempo: volte para a v1, conte as linhas e retorne para a v2 (passo 3)

In [8]:
%%bash
git restore --source=dados-v1 -- data/raw/bike.csv
wc -l data/raw/bike.csv

error: pathspec 'data/raw/bike.csv' did not match any file(s) known to git


13004 data/raw/bike.csv


In [9]:
%%bash
git restore --source=HEAD -- data/raw/bike.csv
wc -l data/raw/bike.csv

error: pathspec 'data/raw/bike.csv' did not match any file(s) known to git


13004 data/raw/bike.csv


### 3. Retreine o pipeline com os dados v2 (o `dvc repro` da T1)

Como o dado mudou, o pipeline inteiro roda de novo. O treino gera um novo run no MLflow e uma nova versão no Registry.

In [10]:
%%bash
dvc repro
dvc metrics show

Verifying data sources in stage: 'data/raw/bike.csv.dvc'

Running stage 'prepare':
> python src/prepare.py
prepare: 13,003 linhas (0 removidas) e 14 colunas -> data/prepared/bike.csv
Updating lock file 'dvc.lock'

Running stage 'split':
> python src/split.py
split: treino 12,331 linhas | teste 672 linhas (2012-06-03 a 2012-06-30)
Updating lock file 'dvc.lock'

Running stage 'train':
> python src/train.py


2026/11/07 22:20:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'bike-demand' already exists. Creating a new version of this model...
2026/11/07 22:20:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bike-demand, version 10
Created version '10' of model 'bike-demand'.


🏃 View run selective-vole-909 at: http://127.0.0.1:5000/#/experiments/1/runs/7f7d44b19ad041a597c267e6cb0ef7ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
run_id=7f7d44b19ad041a597c267e6cb0ef7ad | bike-demand versão 10
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python src/evaluate.py
{
  "rmse": 50.53558727877076,
  "mae": 35.55160367752067,
  "r2": 0.9481765570168661
}
Updating lock file 'dvc.lock'

Running stage 'plot':
> python src/plot.py
plot: reports/previsto_vs_real.png salvo e anexado ao run 7f7d44b19ad041a597c267e6cb0ef7ad
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock data/raw/bike.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
Path          mae      r2       rmse
metrics.json  35.5516  0.94818  50.53559


### 4. Guarde o recibo e publique no GitHub: o código, os ponteiros e as tags (os dados não vão)

In [11]:
%%bash
git add dvc.lock metrics.json reports/previsto_vs_real.png
git commit -m "Retreina pipeline com dados v2"
dvc push
git push -u origin main
git push origin --tags

[main b3b921c] Retreina pipeline com dados v2
 3 files changed, 27 insertions(+), 27 deletions(-)


8 files pushed


fatal: 'origin' does not appear to be a git repository
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
fatal: 'origin' does not appear to be a git repository
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


CalledProcessError: Command 'b'git add dvc.lock metrics.json reports/previsto_vs_real.png\ngit commit -m "Retreina pipeline com dados v2"\ndvc push\ngit push -u origin main\ngit push origin --tags\n'' returned non-zero exit status 128.